# PE-CIR — chạy thử Pic2Word trên Colab
Chạy các ô theo thứ tự. Chọn **Runtime → Change runtime type → GPU** trước.
Notebook giữ CLIP ViT-L/14 đóng băng và train mapper. Chưa train Preserve–Edit vì baseline chưa được xác nhận trên CIRR và Fashion-IQ đầy đủ.

Chuẩn bị: tải `PE-CIR-colab-code.zip` đi kèm. Không cần SSH hay mật khẩu máy chủ.
Đây là smoke test trên tối đa 1.000 ảnh CC3M, seed 0; không dùng kết quả này như tái lập bài báo.
Nguồn: [Pic2Word](https://github.com/google-research/composed_image_retrieval), [CC3M shard](https://huggingface.co/datasets/pixparse/cc3m-wds), [giới hạn Colab](https://research.google.com/colaboratory/faq.html).

## 1. Nạp code đã sửa
Chọn tệp `PE-CIR-colab-code.zip`. Tệp chỉ chứa mã nguồn, cấu hình và kiểm thử; không chứa mật khẩu, dữ liệu hoặc trọng số.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os, subprocess, sys, json
uploaded = files.upload()
bundle_name = "PE-CIR-colab-code.zip"
if bundle_name not in uploaded:
    raise ValueError("Hãy tải lên PE-CIR-colab-code.zip")
repo = Path("/content/PE-CIR")
repo.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[bundle_name])) as archive:
    for item in archive.infolist():
        target = (repo / item.filename).resolve()
        if not target.is_relative_to(repo.resolve()):
            raise ValueError("Đường dẫn ZIP không hợp lệ")
    archive.extractall(repo)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## 2. Kiểm tra GPU và lưu kết quả trên Drive
Colab tự cấp GPU tùy thời điểm. Nếu không có GPU, dừng và đổi loại runtime; không train ViT-L/14 trên CPU.
Cho phép gắn Drive khi Colab hỏi. Mỗi lần thử mới cần RUN_NAME khác; giữ tên khi tiếp tục một lần chạy.

In [ ]:
import torch
from google.colab import drive
assert torch.cuda.is_available(), "Chưa có GPU. Chọn Runtime → Change runtime type → GPU."
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, round(gpu.total_memory / 2**30, 1), "GB VRAM")
try:
    drive.mount("/content/drive", force_remount=True)
except ValueError as error:
    raise RuntimeError(
        "Chưa kết nối được Google Drive. Chạy lại ô này và hoàn tất cửa sổ cấp quyền. "
        "Chưa chạy các ô train phía dưới; kết quả chưa có nơi lưu bền vững."
    ) from error
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Chưa thấy MyDrive. Hãy hoàn tất kết nối Drive trước khi tiếp tục.")
RUN_NAME = "pic2word_cc3m_smoke_seed0_v1"
run_root = Path("/content/drive/MyDrive/PE-CIR/runs") / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
(run_root / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


## 3. Chuẩn bị 1.000 ảnh CC3M
Ô này tải shard đầu tiên (~488 MB) từ bản CC3M WebDataset của pixparse. CLIP sẽ được tải ở bước train và cần thêm dung lượng.
Đây chỉ là mẫu để kiểm tra pipeline, không phải mẫu đại diện cho toàn CC3M. Không đưa triplet CIRR/Fashion-IQ vào manifest train.
Nếu đã có shard trên Drive, điền LOCAL_SHARD để tránh tải lại.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi
LOCAL_SHARD = ""  # ví dụ /content/drive/MyDrive/datasets/cc3m-train-0000.tar
if LOCAL_SHARD:
    shard = Path(LOCAL_SHARD)
    assert shard.is_file(), shard
    source = {"local_shard": str(shard)}
else:
    revision = HfApi().dataset_info("pixparse/cc3m-wds").sha
    shard = Path(hf_hub_download(repo_id="pixparse/cc3m-wds", repo_type="dataset",
        filename="cc3m-train-0000.tar", revision=revision, cache_dir="/content/hf-cache"))
    source = {"dataset": "pixparse/cc3m-wds", "revision": revision, "shard": "cc3m-train-0000.tar"}
subprocess.run([sys.executable, "scripts/prepare_cc3m_shard.py", str(shard), "--max-images", "1000"], check=True)
(run_root / "data_source.json").write_text(json.dumps(source, indent=2))


## 4. Cấu hình lượt thử
Batch 4 là điểm khởi đầu thận trọng, không phải batch đã tối ưu cho mọi GPU. Nếu hết VRAM, dùng RUN_NAME mới và giảm xuống 2. Batch nhỏ chỉ có ít negative; không tương đương batch 1024 bằng cách cộng gradient đơn thuần.
Giữ seed 0 cho lượt này. Không chọn seed dựa trên kết quả tốt/xấu.

In [ ]:
import yaml
config = yaml.safe_load(Path("configs/train_local.yaml").read_text())
config["training"].update(epochs=2, batch_size_per_device=4, max_steps=20,
    warmup_steps=10, seed=0, num_workers=0, save_every_steps=10, purpose="colab_smoke_only")
config["output"] = {"checkpoint_dir": str(run_root / "checkpoints"), "log_dir": str(run_root / "logs")}
config_path = repo / "configs/colab_run.yaml"
config_path.write_text(yaml.safe_dump(config))
subprocess.run([sys.executable, "scripts/train.py", "--config", str(config_path), "--preflight-only"], check=True)


## 5. Train 20 bước và kiểm tra checkpoint
Nếu phiên bị ngắt, chạy lại các ô trước với cùng RUN_NAME. Code sẽ tiếp tục từ checkpoint gần nhất; các bước sau checkpoint cuối có thể cần chạy lại.
Không tự tăng batch khi resume vì sẽ thay đổi cách chia dữ liệu.

In [ ]:
last = run_root / "checkpoints/pic2word/last.pt"
command = [sys.executable, "-u", "scripts/train.py", "--config", str(config_path), "--device", "cuda"]
if last.exists():
    command += ["--resume", str(last)]
subprocess.run(command, check=True)
summary = json.loads((run_root / "logs/pic2word/training_summary.json").read_text())
print(json.dumps(summary, indent=2))
assert last.is_file(), "Chưa lưu được checkpoint"


## 6. Tiếp tục 200 bước khi lượt thử ổn
Đổi RUN_MORE thành True sau khi xem loss hữu hạn và checkpoint đã được lưu. Đây vẫn là kiểm tra kỹ thuật trên tập nhỏ, chưa là screening PE-CIR hay baseline đầy đủ.

In [ ]:
RUN_MORE = False
if RUN_MORE:
    config["training"]["max_steps"] = 200
    config_path.write_text(yaml.safe_dump(config))
    subprocess.run([sys.executable, "-u", "scripts/train.py", "--config", str(config_path),
        "--resume", str(last), "--device", "cuda"], check=True)
else:
    print("Đang giữ lượt thử 20 bước. Đặt RUN_MORE = True để tiếp tục.")


## 6b. Tải nhãn CIRR validation và kiểm tra ảnh
Chạy ô này sau bước 6. Ô này chỉ tải hai tệp nhãn validation, không tải ảnh gốc.
Ảnh CIRR/NLVR2 cần được chuẩn bị theo [hướng dẫn chính thức](https://github.com/Cuberick-Orion/CIRR#raw-images).
Nếu lab đã có bộ ảnh, dùng bộ đó. Không chạy đánh giá khi còn thiếu ảnh.

In [ ]:
from pathlib import Path
import json
from urllib.request import urlopen

CIRR_ROOT = "/content/drive/MyDrive/datasets/cirr"
cirr_root = Path(CIRR_ROOT)
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Hãy kết nối Google Drive ở bước 2 trước.")

base_url = "https://raw.githubusercontent.com/Cuberick-Orion/CIRR/cirr_dataset"
annotation_files = {
    "captions/cap.rc2.val.json": list,
    "image_splits/split.rc2.val.json": dict,
}
for relative_path, expected_type in annotation_files.items():
    destination = cirr_root / relative_path
    if destination.is_file():
        payload = json.loads(destination.read_text(encoding="utf-8"))
    else:
        with urlopen(f"{base_url}/{relative_path}", timeout=60) as response:
            payload = json.load(response)
        if not isinstance(payload, expected_type) or not payload:
            raise ValueError(f"Nhãn không hợp lệ: {relative_path}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
        temporary.replace(destination)
    if not isinstance(payload, expected_type) or not payload:
        raise ValueError(f"Nhãn không hợp lệ: {destination}")

(cirr_root / "img_raw").mkdir(parents=True, exist_ok=True)
from pic2word.data import load_cirr_split
cirr_dataset = load_cirr_split(cirr_root, split="val")
missing_images = cirr_dataset.missing_images()
print("Đã chuẩn bị nhãn CIRR validation.")
print("Số truy vấn:", len(cirr_dataset.queries))
print("Tổng ảnh cần:", len(cirr_dataset.candidate_paths))
print("Số ảnh còn thiếu:", len(missing_images))
if missing_images:
    print("CHƯA SẴN SÀNG đánh giá. Cần bổ sung ảnh CIRR/NLVR2.")
    print("Ví dụ đường dẫn cần có:", missing_images[0])
else:
    print("Đủ tệp ảnh. Có thể chạy mục 7; ảnh sẽ được đọc khi đánh giá.")


## 7. Đánh giá CIRR khi có đủ dữ liệu
Điền CIRR_ROOT trỏ tới thư mục có `captions`, `image_splits`, `img_raw` theo loader. Chỉ đánh giá val; không tune trên test.
Mặc định thiếu ảnh sẽ dừng. Các báo cáo partial cũ không xác nhận baseline. Muốn xác nhận cần CIRR và Fashion-IQ đầy đủ, đối chiếu checkpoint chính thức và protocol tác giả. Notebook này chưa tự chuẩn bị Fashion-IQ.

In [ ]:
CIRR_ROOT = globals().get("CIRR_ROOT", "")  # giữ đường dẫn đã đặt ở bước 6b
if not CIRR_ROOT:
    print("Hãy chạy bước 6b trước để chuẩn bị CIRR.")
else:
    from pic2word.data import load_cirr_split
    cirr_dataset = load_cirr_split(CIRR_ROOT, split="val")
    missing_images = cirr_dataset.missing_images()
    if missing_images:
        raise RuntimeError(f"Còn thiếu {len(missing_images)} ảnh CIRR. Chưa thể đánh giá đầy đủ.")
    subprocess.run([sys.executable, "scripts/evaluate_cirr.py", "--dataset-root", CIRR_ROOT,
        "--checkpoint", str(last), "--config", str(config_path),
        "--index", str(repo / "indexes/colab_cirr_val.pt"),
        "--output", str(run_root / "cirr_val_metrics.json"), "--device", "cuda"], check=True)


## 8. Xem loss và lấy kết quả
Checkpoint, cấu hình đã dùng, nguồn dữ liệu, phiên bản thư viện và CSV nằm trong thư mục RUN_NAME trên Drive.
Gửi `training_summary.json` và `training_metrics.csv` để chọn cấu hình bước tiếp theo. Tốc độ ở lượt 20 bước chỉ là ước tính thô; không cam kết thời lượng train đầy đủ.

In [ ]:
import csv, math
metrics_path = run_root / "logs/pic2word/training_metrics.csv"
with metrics_path.open() as stream:
    rows = list(csv.DictReader(stream))
assert rows and all(math.isfinite(float(row["total_loss"])) for row in rows)
print("Số dòng log:", len(rows), "Loss cuối:", rows[-1]["total_loss"])
print("Kết quả trên Drive:", run_root)
